# Convert vishub-projects.csv to Project Markdown Files

Reads `vishub-projects.csv` and writes one `.md` file per row into `../_project/`.

Each file follows the frontmatter format used by existing project pages (see `_project/peacerep.md`).

In [17]:
import pandas as pd
import re
from pathlib import Path

df = pd.read_csv('vishub-projects.csv', encoding='utf-8')

df.columns = [
    'id', 'start_time', 'completion_time', 'email', 'submitter_name', 'last_modified',
    'title', 'description', 'people', 'student_led', 'tags', 'body',
    'partners', 'website', 'outputs', 'publications', 'thumbnail_url',
    'thumbnail_filename', 'additional_photos'
]

print(f"Loaded {len(df)} project(s)")
df[['title', 'people', 'tags']].head()

Loaded 1 project(s)


,title,people,tags
0,Digital Ghosts,"Dorsey Kaufmann, Andrea Kocsis","Digital Humanities, Uncertainty Visualization,..."


In [18]:
from urllib.parse import unquote, urlparse

def slugify(text):
    """Convert a title to a filename-safe slug (no spaces or special chars)."""
    text = str(text).lower()
    text = re.sub(r'[^\w\s-]', '', text)
    text = re.sub(r'[\s_]+', '', text)
    return text.strip('-')


def parse_comma_list(text):
    """Split a comma-separated string into a stripped list, skipping empties."""
    if pd.isna(text) or not str(text).strip():
        return []
    return [item.strip() for item in str(text).split(',') if item.strip()]


def parse_bullet_list(text):
    """Extract items from a bullet list, stripping leading bullet characters."""
    if pd.isna(text) or not str(text).strip():
        return []
    items = []
    for line in str(text).splitlines():
        line = line.strip()
        # Strip leading bullet chars including ¥ (\xa5) used by Microsoft Forms
        line = re.sub(r'^[-•●•●¥]\s*', '', line)
        line = line.strip('"').strip()
        if line:
            items.append(line)
    return items


def linkify(text):
    """Convert bare URLs to markdown links.
    If the text ends with a URL, uses the preceding text as the link label.
    Otherwise wraps each URL inline as [url](url)."""
    text = text.strip()
    m = re.match(r'^(.+?)\s+(https?://\S+)\s*$', text)
    if m:
        return f'[{m.group(1).strip()}]({m.group(2)})'
    return re.sub(r'(https?://\S+)', r'[\1](\1)', text)


def get_image_path(thumbnail_filename, thumbnail_url):
    """Build the image path from the 'File name for thumbnail' column.
    If the filename has no extension, infers it from the thumbnail URL."""
    name = str(thumbnail_filename).strip() if not pd.isna(thumbnail_filename) else ''
    if not name or name == 'nan':
        return ''
    if '.' not in name.split('/')[-1]:
        url = str(thumbnail_url).strip() if not pd.isna(thumbnail_url) else ''
        m = re.search(r'\.(png|jpg|jpeg|gif|svg|webp)', url, re.IGNORECASE)
        if m:
            name = name + '.' + m.group(1).lower()
    return f"figures/{name}"


def parse_additional_photos(text):
    """Extract filenames from semicolon-separated SharePoint (or local) URLs/paths."""
    if pd.isna(text) or not str(text).strip():
        return []
    filenames = []
    for entry in str(text).split(';'):
        entry = entry.strip()
        if not entry:
            continue
        # If it looks like a URL, extract and decode just the filename
        if entry.startswith('http'):
            filename = unquote(urlparse(entry).path.split('/')[-1])
        else:
            filename = entry
        if filename:
            filenames.append(filename)
    return filenames


def row_to_markdown(row):
    title       = str(row['title']).strip()
    description = str(row['description']).strip() if not pd.isna(row['description']) else ''
    body        = str(row['body']).strip()        if not pd.isna(row['body'])        else ''
    partners    = str(row['partners']).strip()    if not pd.isna(row['partners'])    else ''
    website     = str(row['website']).strip()     if not pd.isna(row['website'])     else ''
    student_led = str(row['student_led']).strip() if not pd.isna(row['student_led']) else ''

    people            = parse_comma_list(row['people'])
    outputs           = [linkify(item) for item in parse_bullet_list(row['outputs'])]
    tags              = parse_comma_list(row['tags'])
    publications      = parse_bullet_list(row['publications'])
    additional_photos = parse_additional_photos(row['additional_photos'])

    image_path = get_image_path(row['thumbnail_filename'], row['thumbnail_url'])

    lines = ['---']
    lines.append('layout: project')
    lines.append(f'title: {title}')
    if image_path:
        lines.append(f'image: {image_path}')
    lines.append(f'description: {description}')

    if people:
        lines.append('people:')
        for p in people:
            lines.append(f'- {p}')

    if outputs:
        lines.append('outputs:')
        for item in outputs:
            lines.append(f'- "{item.replace(chr(34), chr(92) + chr(34))}"')

    if tags:
        lines.append('tags:')
        for t in tags:
            lines.append(f'- {t}')

    if publications:
        lines.append('publications:')
        for pub in publications:
            lines.append(f'- "{pub}"')

    if partners:
        lines.append(f'partners: {partners}')

    if website:
        lines.append(f'website: {website}')

    if student_led.lower() == 'yes':
        lines.append('student_led: true')

    if additional_photos:
        lines.append('additional_photos:')
        for photo in additional_photos:
            lines.append(f'- {photo}')

    lines.append('---')
    lines.append('')
    lines.append(body)

    return '\n'.join(lines) + '\n'

In [19]:
output_dir = Path('../_project')
output_dir.mkdir(exist_ok=True)

for _, row in df.iterrows():
    title = str(row['title']).strip()
    if not title or title == 'nan':
        print(f"  Skipping row {row['id']}: no title")
        continue

    slug = slugify(title)
    filepath = output_dir / f'{slug}.md'
    content = row_to_markdown(row)

    filepath.write_text(content, encoding='utf-8')
    print(f"  Written: {filepath}")

print("\nDone.")

  Written: ../_project/digitalghosts.md

Done.


In [20]:
# Preview the first generated file
generated = list(output_dir.glob('*.md'))
if generated:
    print(generated[0].read_text(encoding='utf-8'))

---
layout: project
title: Vistorian
image: figures/vistorian.png
description: The Vistorian is an online tool for interactive exploration of dynamic, multivariate, and geographic networks
people:
- Benjamin Bach
- Xinhuan Shu
- Alexis Pister
- Nathalie Henry Riche
- Roland Fernandez
- Emmanoulis Giannisakis
- Bongshin Lee
- Jean-Daniel Fekete
tags:
- Network
---
**Partners**: Inria, Microsoft

**Link**: [Vistorian](https://vistorian.github.io/vistorian/)

The Vistorian is an online tool for interactive exploration of dynamic, multivariate, and geographic networks. Main features include a wide range of different interactive network visualizations: node link diagrams, side-by-side views and parallel use of any of these visualizations. and a geocoding service that obtains geographic locations from placenames in your dataset.


